In [2]:
import math 
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
class Value:
    index = 0
    def __init__(self, data, _children=(), _op='', label=None):
        self.data = data
        if label:
            self.label = label
        else :
            self.label =f"v_{Value.index}"
            Value.index += 1
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set[Value](_children)
        self._op = _op

    def __repr__(self):
        #return f"Value({self.label}: data={self.data}), grad={round(self.grad,10)}, op={self._op}"
        return f"Value({self.label}: data={self.data}), grad={self.grad}, op={self._op}"

    def __add__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        out =  Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + other*(-1)

    def __rsub__(self, other):
        return self*(-1) + other

    def __mul__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out        

    def __rmul__(self, other):
        return self * other

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int and float are supported"
        out =  Value(self.data**other, (self,), f'**{other}')
        def _backward():
            self.grad += out.grad* other*self.data**(other-1)
        out._backward = _backward
        return out   

    def __truediv__(self, other):
        return self * other**-1

    def __rtruediv__(self, other):
        return other * self**-1

    def tanh(self):
        out =  Value(math.tanh(self.data), (self,), 'tanh')
        def _backward():
            self.grad += out.grad/(math.cosh(self.data)**2)
        out._backward = _backward
        return out   

    def exp(self):
        out =  Value(math.exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.grad*math.exp(self.data)
        out._backward = _backward
        return out   

    def print_chain(self):
        if self._prev:
            for p in self._prev:
                p.print_chain()
        print(self)

    
    def backward(self) -> None:
        # topological sort the nodes, and zero grads so a new backward
        # does not add onto the previous one
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                v.grad = 0.0
                topo.append(v)
        build_topo(self)

        # backward pass
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
                    



In [4]:
"""
# input layer
a = Value(3.0, label='a')
b = a.tanh()
b.label = "b"
b.backward()
print("--------------------------------")
b.print_chain()


x = Value(3.0, label='x')
tmp = (-1)*x
# y = x.exp() - tmp.exp()  OK
# y = x.exp() + tmp.exp()  OK
# y =(tmp**2)*(tmp**(-1)) OK
y = (x.exp() - tmp.exp()) / (x.exp() + tmp.exp())  # OK
y.label = "y"
y.backward()
print("--------------------------------")
y.print_chain()



a = Value(3.0, label='a')
b = 2*a
b.label = "b"
c = (3*b) - (2*b)
c.label = "c"
c.backward()
print("--------------------------------")
c.print_chain()


a = Value(3.0, label='a')
b = a+a; b.label = "b"
b.backward()
print("--------------------------------")
b.print_chain()

a = Value(-2.0, label='a')
b = Value(3.0, label='b')
d = a*b; d.label = "d"
e = a + b; e.label = "e"
f = d*e; f.label = "f"
f.backward()
print("--------------------------------")
f.print_chain()
"""

x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

b = Value(6.8813735870195432, label='b')

n = x1*w1 + x2*w2 + b; n.label = "n"
o = n.tanh()
o.backward()
print("--------------------------------")
o.print_chain()













--------------------------------
Value(b: data=6.881373587019543), grad=0.4999999999999999, op=
Value(w2: data=1.0), grad=0.0, op=
Value(x2: data=0.0), grad=0.4999999999999999, op=
Value(v_1: data=0.0), grad=0.4999999999999999, op=*
Value(x1: data=2.0), grad=-1.4999999999999996, op=
Value(w1: data=-3.0), grad=0.9999999999999998, op=
Value(v_0: data=-6.0), grad=0.4999999999999999, op=*
Value(v_2: data=-6.0), grad=0.4999999999999999, op=+
Value(n: data=0.8813735870195432), grad=0.4999999999999999, op=+
Value(v_4: data=0.7071067811865476), grad=1.0, op=tanh


In [5]:
import torch
x1 = torch.Tensor([2]).double()                         ; x1.requires_grad = True
x2 = torch.Tensor([0]).double()                         ; x2.requires_grad = True   

w1 = torch.Tensor([-3]).double()                        ; w1.requires_grad = True
w2 = torch.Tensor([1]).double()                         ; w2.requires_grad = True

b = torch.Tensor([6.8813735870195432]).double()         ; b.requires_grad = True

n = x1*w1 + x2*w2 + b; n.label = "n"
o = torch.tanh(n)
print(o.data.item())

o.backward()
print("--------------------------------")
print(x1.grad.item())
print(x2.grad.item())
print(w1.grad.item())
print(w2.grad.item())
print(b.grad.item())




0.7071066904050358
--------------------------------
-1.5000003851533106
0.5000001283844369
1.0000002567688737
0.0
0.5000001283844369


In [17]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        act = sum((xi*wi for xi,wi in zip(x, self.w)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]







In [461]:


"""
mpl = MLP(3, [4,4,1])
x = [2,3,-1]
print(mpl(x))
"""

xs = [
    [2.0, 3.0, -1],
    [3.0, -1, 0.5],
    [0.5, 1, 1],
    [1, 1, -1]
]
ys = [1, -1, -1, 1]

n = MLP(3, [4,4,1])


In [ ]:
for k in range(60):    
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for yout, ygt in zip(ypred, ys))

    # zero gradients
    for p in n.parameters():
        p.grad = 0.0

    # backward pass
    loss.backward()

    # update parameters
    for p in n.parameters():
        p.data += -0.1* p.grad

    # print loss
    print(loss)


Value(v_123592: data=7.8509923121640846), grad=1.0, op=+
Value(v_123949: data=5.214205361779463), grad=1.0, op=+
Value(v_124306: data=2.480235863849964), grad=1.0, op=+
Value(v_124663: data=1.4944142138996044), grad=1.0, op=+
Value(v_125020: data=0.838998112672209), grad=1.0, op=+
Value(v_125377: data=0.5133348468873429), grad=1.0, op=+
Value(v_125734: data=0.28830388980365673), grad=1.0, op=+
Value(v_126091: data=0.21316943468108315), grad=1.0, op=+
Value(v_126448: data=0.16913700883540667), grad=1.0, op=+
Value(v_126805: data=0.13960126164867015), grad=1.0, op=+
Value(v_127162: data=0.11852475272922183), grad=1.0, op=+
Value(v_127519: data=0.10278297776553494), grad=1.0, op=+
Value(v_127876: data=0.09060735655686551), grad=1.0, op=+
Value(v_128233: data=0.08092634920853989), grad=1.0, op=+
Value(v_128590: data=0.07305496516411487), grad=1.0, op=+
Value(v_128947: data=0.06653590543118293), grad=1.0, op=+
Value(v_129304: data=0.06105275207006372), grad=1.0, op=+
Value(v_129661: data=0.

In [454]:
ypred

[Value(v_101379: data=0.9999980417592732), grad=-3.9164814535919135e-06, op=tanh,
 Value(v_101464: data=-0.9999758539122494), grad=4.8292175501174484e-05, op=tanh,
 Value(v_101549: data=-0.9999734639580842), grad=5.3072083831651184e-05, op=tanh,
 Value(v_101634: data=0.9999983722383056), grad=-3.255523388867587e-06, op=tanh]